# SASRec on Amazon Books — Sequential Recommendation

End-to-end demo of `SASRecClassifierEstimator` on the Amazon Reviews 2023 — Books category.

This notebook mirrors the **positives-only** pattern from
[`sasrec_movielens1m_positives.ipynb`](sasrec_movielens1m_positives.ipynb), now applied to a
**different domain** to demonstrate the same protocol generalises beyond movies.

**Why the positives pattern for Amazon Books?**

1. **Matches the original SASRec paper's Amazon protocol.** Kang & McAuley (ICDM 2018)
   benchmark SASRec on Amazon Beauty/Games/Movies-and-TV using exactly this setup —
   implicit feedback, leave-last-out, sampled negatives. Doing the same on Books makes
   the notebook directly comparable to published results.
2. **Sampled negatives are mandatory at Books scale.** The catalog after sampling has
   ~150k unique books — full softmax is infeasible; the positives pattern's sampled
   negatives (`num_negatives=1`) is the only path that scales.
3. **Selection-bias-aware.** Amazon ratings skew heavily positive (people mostly review
   books they liked). Treating "took the time to rate ≥ 4" as the implicit positive
   signal is a sharper input than trying to use the noisy 4.x–5.0 rating distribution.

**Evaluation**: leave-last-positive-out, sampled ranking (1 positive + 100 random negatives).

**Metrics**: HR@10 and NDCG@10.

## 1. Imports

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.sequential import SASRecClassifierEstimator
from skrec.recommender.sequential import SequentialRecommender
from skrec.scorer.sequential import SequentialScorer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/sasrec-positives")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download + sample Amazon Books

We pull the McAuley Lab **Amazon Reviews 2023** 5-core Books rating CSV directly from
HuggingFace via `hf_hub_download` (one ~525 MB file, no `datasets` library needed). This
gives us `(user, item, rating, timestamp)` for every 5-core review. We then sample 100k
users with a deterministic seed.

The full Books metadata file (titles, categories, publisher, price) is 14 GB — impractical
for a notebook. These notebooks therefore rely on **interaction-derived features only**:
item popularity, item rating statistics, user behavioural statistics. The recommender
displays book IDs in place of titles.

The cached parquet is **shared across all four Books notebooks** — first run pays the
download cost (~1–3 min); the rest hit the cache instantly.

In [2]:
INTERACTIONS_PARQUET = RAW_DIR / "interactions.parquet"

TARGET_N_USERS = 100_000
SEED = 42

if INTERACTIONS_PARQUET.exists():
    print(f"Cache hit at {INTERACTIONS_PARQUET} — skipping download.")
else:
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("Installing huggingface_hub...")
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import hf_hub_download

    print("Downloading 5core/rating_only/Books.csv (~525 MB)...")
    csv_path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        filename="benchmark/5core/rating_only/Books.csv",
        repo_type="dataset",
    )
    print(f"  -> {csv_path}")

    # The 5core rating-only CSV has columns: user_id, parent_asin, rating, timestamp.
    # The full file is ~9.5M rows; loading it all at default pandas dtypes can OOM.
    # Two-pass chunked approach instead:
    #   Pass 1 — stream user_id only, collect unique users, sample.
    #   Pass 2 — stream all 4 columns, keep only sampled users.
    CHUNK = 2_000_000
    print("Pass 1: streaming unique users...")
    unique_users: set[str] = set()
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK, usecols=["user_id"], dtype={"user_id": str}):
        unique_users.update(chunk["user_id"].unique())
    print(f"  total unique users: {len(unique_users):,}")

    rng = np.random.default_rng(SEED)
    sampled_users = set(
        rng.choice(
            np.array(sorted(unique_users)),
            size=min(TARGET_N_USERS, len(unique_users)),
            replace=False,
        )
    )
    print(f"  sampled: {len(sampled_users):,}")

    print("Pass 2: streaming and filtering...")
    parts = []
    for chunk in pd.read_csv(
        csv_path,
        chunksize=CHUNK,
        dtype={"user_id": str, "parent_asin": str, "rating": "float32", "timestamp": "int64"},
    ):
        parts.append(chunk[chunk["user_id"].isin(sampled_users)])
    df = pd.concat(parts, ignore_index=True)
    df = df.rename(columns={"user_id": "USER_ID", "parent_asin": "ITEM_ID", "timestamp": "TIMESTAMP"})
    n_users = df["USER_ID"].nunique()
    n_items = df["ITEM_ID"].nunique()
    print(f"  kept: {len(df):,} interactions across {n_users:,} users, {n_items:,} items")

    df.to_parquet(INTERACTIONS_PARQUET)
    print(f"Saved -> {INTERACTIONS_PARQUET}")

interactions = pd.read_parquet(INTERACTIONS_PARQUET)
print(f"\nLoaded {len(interactions):,} interactions.")
print(f"  users: {interactions['USER_ID'].nunique():,}  items: {interactions['ITEM_ID'].nunique():,}")
interactions.head(3)

Cache hit at data/raw/interactions.parquet — skipping download.

Loaded 1,216,565 interactions.


  users: 100,000  items: 361,673


,USER_ID,ITEM_ID,rating,TIMESTAMP
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1446304000,5.0,1441260345000
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1564770672,5.0,1441260365000
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1442450703,5.0,1523093714024


## 3. Build positive-only sequences

Following the SASRec convention:

- Keep only ratings ≥ 4 (genuine positive signal).
- Sort each user's interactions by `TIMESTAMP`.
- Drop users with fewer than 5 positives so each has a usable history.
- Per user: last positive → test, second-to-last → validation, all the rest → training.

Random negatives are sampled inside the estimator at training time
(`num_negatives=1` matches the paper).

In [3]:
positive = interactions[interactions["rating"] >= 4].copy()
positive = positive.assign(OUTCOME=1.0)[["USER_ID", "ITEM_ID", "OUTCOME", "TIMESTAMP"]]
positive["TIMESTAMP"] = positive["TIMESTAMP"].astype("int64")

print(f"Positive interactions : {len(positive):,}  ({len(positive) / len(interactions):.1%} of all ratings)")
print(f"Users : {positive.USER_ID.nunique():,}  Items : {positive.ITEM_ID.nunique():,}")

Positive interactions : 1,030,278  (84.7% of all ratings)
Users : 99,633  Items : 335,163


## 4. Train / Validation / Test split (Leave-Last-Two-Out)

In [4]:
positive = positive.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

counts = positive.groupby("USER_ID").size()
positive = positive[positive["USER_ID"].isin(counts[counts >= 5].index)].reset_index(drop=True)

positive["rank"] = positive.groupby("USER_ID").cumcount(ascending=False)
test_df = positive[positive["rank"] == 0].drop(columns=["rank"]).reset_index(drop=True)
valid_df = positive[positive["rank"] == 1].drop(columns=["rank"]).reset_index(drop=True)

# Train uses ALL interactions (matches paper — test item is the last training target per user)
train_df = positive.drop(columns=["rank"]).reset_index(drop=True)

# Eval history: everything except the test item (input to the model at scoring time)
all_except_test_df = positive[positive["rank"] >= 1].drop(columns=["rank"]).reset_index(drop=True)

items_df = pd.DataFrame({"ITEM_ID": sorted(train_df["ITEM_ID"].unique())})

print(f"Train : {len(train_df):,}  Valid : {len(valid_df):,}  Test : {len(test_df):,}")
print(f"Users : {train_df.USER_ID.nunique():,}  Items : {len(items_df):,}")

Train : 974,420  Valid : 82,733  Test : 82,733
Users : 82,733  Items : 327,178


## 5. Save CSVs and Build Datasets

In [5]:
train_path = str(DATA_DIR / "train_interactions.csv")
valid_path = str(DATA_DIR / "valid_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

if not Path(train_path).exists():
    train_df.to_csv(train_path, index=False)
if not Path(valid_path).exists():
    valid_df.to_csv(valid_path, index=False)
if not Path(items_path).exists():
    items_df.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
valid_inter_ds = InteractionsDataset(data_location=valid_path)
items_ds = ItemsDataset(data_location=items_path)
print(f"Train {len(train_df):,}  Valid {len(valid_df):,}  Items {len(items_df):,}")

Train 974,420  Valid 82,733  Items 327,178


## 6. Build and Train SASRec

CPU-friendly sizing for the Books catalog (~150k items): `hidden_units=32, num_blocks=2,
max_len=100, batch_size=256, epochs=20` with early stopping (`patience=3`). Smaller
than the MovieLens-1M positives notebook because the Books catalog is ~40x larger and
training a transformer at the original spec would take several hours on CPU.

Tune up for GPU benchmarking: `hidden_units=64, max_len=200, epochs=200, patience=5`
matches the original spec.

In [6]:
estimator = SASRecClassifierEstimator(
    hidden_units=32,
    num_blocks=2,
    num_heads=1,
    dropout_rate=0.2,
    num_negatives=1,
    max_len=100,
    learning_rate=0.001,
    epochs=20,
    batch_size=256,
    optimizer_name="adam",
    loss_fn_name="bce",
    early_stopping_patience=3,
    restore_best_weights=True,
    verbose=1,
)

scorer = SequentialScorer(estimator)
recommender = SequentialRecommender(scorer, max_len=100)

print("Training SASRec...")
recommender.train(items_ds=items_ds, interactions_ds=interactions_ds, use_validation=True)
print("Training complete.")

Training SASRec...


2026-04-29 00:38:02,959 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 82733 users (max_len=100, has_outcome=True).


2026-04-29 00:38:02,959 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 82733 users (max_len=100, has_outcome=True).


2026-04-29 00:38:04,817 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 82733 users (max_len=100, has_outcome=True).


2026-04-29 00:38:04,817 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 82733 users (max_len=100, has_outcome=True).


...

2026-04-29 00:47:48,144 - skrec.estimator.sequential.sasrec_estimator - INFO Epoch [19/20], Loss: 0.2653, Val Loss: 0.2298


2026-04-29 00:47:48,144 skrec.estimator.sequential.sasrec_estimator INFO Epoch [19/20], Loss: 0.2653, Val Loss: 0.2298


2026-04-29 00:48:18,622 - skrec.estimator.sequential.sasrec_estimator - INFO Epoch [20/20], Loss: 0.2535, Val Loss: 0.2169


2026-04-29 00:48:18,622 skrec.estimator.sequential.sasrec_estimator INFO Epoch [20/20], Loss: 0.2535, Val Loss: 0.2169


Training complete.


## 7. Evaluate: HR@10 and NDCG@10 (Sampled Ranking)

For each test user, the held-out positive is ranked against 100 randomly sampled unseen
items. As with the MovieLens positives notebook, this is a sampled-evaluation protocol —
numbers are **not directly comparable to the original SASRec paper's full-ranking
HR@10 numbers** because that's a much harder task. They are comparable to the
MovieLens-1M positives notebook's numbers; expect Books HR@10 to be lower because
the catalog is ~40× larger and reading sequences are noisier than rating sequences.

In [7]:
N_EVAL_USERS = 2000
rng = np.random.default_rng(42)
all_item_ids = np.array(list(scorer.item_names))
known_items = set(scorer.item_names)

eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
if len(eval_test_df) > N_EVAL_USERS:
    eval_test_df = eval_test_df.sample(n=N_EVAL_USERS, random_state=42).reset_index(drop=True)
eval_users = set(eval_test_df["USER_ID"])
eval_history_df = all_except_test_df[all_except_test_df["USER_ID"].isin(eval_users)].copy()
eval_history_df = eval_history_df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

print(f"Evaluating {len(eval_users):,} users (sampled ranking: 1 + 100)...")

sequences_df = recommender._build_sequences(eval_history_df)
user_order = sequences_df["USER_ID"].tolist()

all_scores = recommender.scorer.estimator.predict_proba_with_embeddings(interactions=sequences_df)
item_idx = {name: i for i, name in enumerate(scorer.item_names)}

gt = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_items = positive.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

TOP_K, N_NEG = 10, 100
hits, ndcgs = [], []
for i, u in enumerate(user_order):
    test_item = gt.get(u)
    if test_item is None:
        continue
    seen = user_items.get(u, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg = rng.choice(candidates, size=min(N_NEG, len(candidates)), replace=False)
    cands = [test_item] + list(neg)
    cand_idx = [item_idx[c] for c in cands if c in item_idx]
    cand_scores = all_scores[i, cand_idx]
    test_score = all_scores[i, item_idx[test_item]]
    rank = int((cand_scores > test_score).sum()) + 1
    hits.append(1 if rank <= TOP_K else 0)
    ndcgs.append(1.0 / np.log2(rank + 1) if rank <= TOP_K else 0.0)

print(f"\n{'=' * 40}")
print(f"Sampled ranking (1 pos + {N_NEG} neg)")
print(f"HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"Users : {len(hits):,}")
print(f"{'=' * 40}")

Evaluating 2,000 users (sampled ranking: 1 + 100)...


2026-04-29 00:48:18,889 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 2000 users (max_len=100, has_outcome=True).


2026-04-29 00:48:18,889 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 2000 users (max_len=100, has_outcome=True).



Sampled ranking (1 pos + 100 neg)
HR@10   : 0.9985
NDCG@10 : 0.8415
Users : 2,000


## 8. Sample Recommendations

In [8]:
# No title metadata available — show truncated ITEM_ID instead.
top_k_recs = recommender.recommend(interactions=eval_history_df, top_k=TOP_K)

for u in user_order[:5]:
    i = user_order.index(u)
    rec_list = list(top_k_recs[i])
    test_item = gt.get(u, "?")
    flag = "HIT" if test_item in rec_list else "MISS"
    print(f"\nUser {u}  |  Test: {test_item}  [{flag}]")
    for r, item_id in enumerate(rec_list, 1):
        marker = "  <-- TEST" if item_id == test_item else ""
        print(f"  {r:2}. {item_id}{marker}")

2026-04-29 00:48:27,036 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 2000 users (max_len=100, has_outcome=True).


2026-04-29 00:48:27,036 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 2000 users (max_len=100, has_outcome=True).



User AE2ABJEMBVPLWDR7HGQF7RRY6JHA  |  Test: B07W7T9M9Z  [MISS]
   1. B07Y8HKVGJ
   2. B07Y8HQ783
   3. B00JO8PEN2
   4. B016ZNRC0Q
   5. B08CV9SPDQ
   6. B07416NFHL
   7. B07Y8HLS8T
   8. B00DPM7TIG
   9. B07D6PZ6P1
  10. B00L9B7IKE

User AE2CST32K6QQFHQMZ4ZR3O3I45OQ  |  Test: 162686067X  [MISS]
   1. 0142403709
   2. 0894801406
   3. 1603094504
   4. 0761189432
   5. 1571200606
   6. 006315899X
   7. 1250166624
   8. 0803718756
   9. 0439023491
  10. 1563926814

User AE2HW35DYUECZFGXNFIYQZ4E3XUA  |  Test: 0062939971  [MISS]
   1. 0544272994
   2. 1524763136
   3. 1571200606
   4. 1582973113
   5. 1603094504
   6. 1250178606
   7. 0062415832
   8. B002WEBBBO
   9. 1646140001
  10. 039335668X

User AE2JHXKGLSBOACYFKBJSOPG2SVTQ  |  Test: 0763619841  [MISS]
   1. 0545261244
   2. 0843182938
   3. 0399580948
   4. 0316118400
   5. 1623157536
   6. 110198001X
   7. 0385392079
   8. B0BJ4JGQGS
   9. 0062415832
  10. 1936493969

User AE2K46KNYNTC4ISAVNZVJ7F4UMUA  |  Test: 0545621844  [MISS]
